# 04 – scikit-learn: Ein erstes diagnostisches Vorhersagemodell

**scikit-learn** (kurz *sklearn*) ist die wichtigste Python-Bibliothek für klassisches **maschinelles Lernen**. In diesem Notebook trainieren wir ein Modell, das anhand von Zellkern-Merkmalen aus einer Feinnadelbiopsie vorhersagt, ob ein Brusttumor **gutartig** oder **bösartig** ist.

### Lernziele
Nach diesem Notebook können Sie
- das Grundprinzip des *überwachten Lernens* erklären,
- Daten in Trainings- und Testdaten aufteilen – und begründen, warum,
- ein Modell mit dem sklearn-Muster `fit` → `predict` trainieren,
- ein Modell mit **Sensitivität, Spezifität**, Konfusionsmatrix und ROC-Kurve bewerten,
- typische Fallstricke (Overfitting, falscher Schwellenwert) erkennen.

### Hinweise
- Zellen der Reihe nach mit `Shift + Enter` ausführen.
- ⭐ = wird in der Vorlesung gezeigt, Rest zur Vertiefung.
- Viele Begriffe kennen Sie bereits aus der Biometrie bzw. Epidemiologie: Die **Vierfeldertafel** heißt hier *Konfusionsmatrix*.

## ⭐ 1. Was ist überwachtes Lernen?

Die Idee ähnelt der klinischen Ausbildung: Man zeigt einem System viele **Beispiele mit bekannter Diagnose** (die „Lösung“), und es lernt daraus Regeln, um **neue, unbekannte Fälle** einzuordnen.

| Begriff | Bedeutung | In unserem Beispiel |
|---|---|---|
| **Features** `X` | Eingabevariablen, Prädiktoren | 30 Messwerte der Zellkerne (Radius, Textur, …) |
| **Zielvariable** `y` | Was vorhergesagt werden soll | gutartig (0) / bösartig (1) |
| **Training** | Modell lernt aus Beispielen | `model.fit(X_train, y_train)` |
| **Vorhersage** | Modell wendet Gelerntes an | `model.predict(X_test)` |

Ist `y` eine Kategorie, spricht man von **Klassifikation**; ist `y` eine Zahl (z. B. Blutdruck), von **Regression**.

## ⭐ 2. Daten laden

Der *Breast Cancer Wisconsin*-Datensatz ist in sklearn bereits enthalten (keine Internetverbindung nötig). Er enthält 569 Biopsien; für jede wurden Merkmale der Zellkerne aus digitalisierten Mikroskopiebildern berechnet.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

daten = load_breast_cancer(as_frame=True)
X = daten.data

# Achtung: In sklearn ist 0 = bösartig, 1 = gutartig.
# Für die medizinische Intuition drehen wir das um: 1 = bösartig ("positiver Befund").
y = 1 - daten.target

print("Form von X:", X.shape)
print(y.value_counts().rename({0: "gutartig (0)", 1: "bösartig (1)"}))

> **Merke:** Prüfen Sie immer, wie die Zielvariable kodiert ist! Ob „1“ krank oder gesund bedeutet, entscheidet darüber, was Sensitivität und Spezifität später aussagen.

In [ ]:
X.iloc[:5, :6]    # die ersten 5 Biopsien, die ersten 6 Merkmale

Ein kurzer Blick, ob sich die Gruppen in einzelnen Merkmalen überhaupt unterscheiden – das kennen Sie aus Notebook 03:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, merkmal in zip(axes, ["mean radius", "mean concave points"]):
    ax.hist(X.loc[y == 0, merkmal], bins=25, alpha=0.6, label="gutartig")
    ax.hist(X.loc[y == 1, merkmal], bins=25, alpha=0.6, label="bösartig", color="crimson")
    ax.set_xlabel(merkmal)
    ax.set_ylabel("Anzahl")
    ax.legend()
fig.tight_layout()
plt.show()

Die Verteilungen überlappen, unterscheiden sich aber deutlich. Ein Modell soll nun **alle 30 Merkmale gleichzeitig** nutzen, um möglichst gut zu trennen.

## ⭐ 3. Trainings- und Testdaten

Die wichtigste Regel im maschinellen Lernen: **Ein Modell darf nie auf den Daten bewertet werden, mit denen es trainiert wurde.** Das wäre wie eine Prüfung, deren Fragen man vorher auswendig gelernt hat – gute Note, aber nichts über das echte Können ausgesagt.

Deshalb legen wir einen Teil der Daten (hier 25 %) als **Testdaten** beiseite und schauen sie erst ganz am Ende an.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25 % für den Test
    stratify=y,          # gleicher Anteil bösartiger Fälle in beiden Teilen
    random_state=42,     # reproduzierbare Aufteilung
)

print("Training:", X_train.shape[0], "Biopsien")
print("Test:    ", X_test.shape[0], "Biopsien")

## ⭐ 4. Das sklearn-Grundmuster am Beispiel eines Entscheidungsbaums

Alle Modelle in sklearn funktionieren nach demselben Muster:

```python
modell = Modellklasse(einstellungen)   # 1. Modell erzeugen
modell.fit(X_train, y_train)           # 2. trainieren
vorhersage = modell.predict(X_test)    # 3. vorhersagen
```

Wir beginnen mit einem **Entscheidungsbaum** – er ist leicht zu verstehen, weil er wie ein klinischer Diagnose-Algorithmus aus Ja/Nein-Fragen aufgebaut ist.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

baum = DecisionTreeClassifier(max_depth=3, random_state=42)   # 1. erzeugen
baum.fit(X_train, y_train)                                    # 2. trainieren
vorhersage_baum = baum.predict(X_test)                        # 3. vorhersagen

print("Erste 15 Vorhersagen:", vorhersage_baum[:15])
print("Wahre Diagnosen:     ", y_test.values[:15])

Der Vorteil eines Baumes: Wir können uns die gelernten Regeln ansehen.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(baum, feature_names=X.columns, class_names=["gutartig", "bösartig"],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.show()

**So lesen Sie den Baum:** Oben beginnen, die Bedingung prüfen; wenn wahr → links, wenn falsch → rechts. `samples` = Anzahl Trainingsfälle in diesem Knoten, `value` = [gutartig, bösartig], `class` = Mehrheitsentscheidung.

## ⭐ 5. Modellbewertung: Die Vierfeldertafel

Die **Genauigkeit** (*accuracy*) ist der Anteil korrekter Vorhersagen. Sie allein reicht in der Medizin aber nicht: Ein übersehenes Karzinom (falsch negativ) ist etwas anderes als eine unnötige Nachuntersuchung (falsch positiv).

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print(f"Genauigkeit: {accuracy_score(y_test, vorhersage_baum):.1%}")

ConfusionMatrixDisplay.from_predictions(
    y_test, vorhersage_baum, display_labels=["gutartig", "bösartig"], cmap="Blues"
)
plt.xlabel("Vorhersage des Modells")
plt.ylabel("Wahre Diagnose")
plt.show()

Aus der Konfusionsmatrix berechnen wir die bekannten Gütekriterien diagnostischer Tests:

|  | Vorhersage: gutartig | Vorhersage: bösartig |
|---|---|---|
| **wahr: gutartig** | richtig negativ (TN) | falsch positiv (FP) |
| **wahr: bösartig** | falsch negativ (FN) | richtig positiv (TP) |

Da wir diese Berechnung gleich mehrmals brauchen, schreiben wir eine kleine **Funktion**:

In [ ]:
def guetekriterien(y_wahr, y_vorhersage):
    tn, fp, fn, tp = confusion_matrix(y_wahr, y_vorhersage).ravel()
    return pd.Series({
        "Sensitivität": tp / (tp + fn),   # Anteil erkannter Kranker
        "Spezifität":   tn / (tn + fp),   # Anteil korrekt erkannter Gesunder
        "PPV":          tp / (tp + fp),   # positiver prädiktiver Wert
        "NPV":          tn / (tn + fn),   # negativer prädiktiver Wert
        "Falsch neg.":  fn,
        "Falsch pos.":  fp,
    })

guetekriterien(y_test, vorhersage_baum).round(3)

> **Genau hinsehen!** Eine Genauigkeit von knapp 90 % klingt gut. Schauen Sie aber auf die Sensitivität: Der einfache Baum übersieht etwa **jeden vierten bösartigen Tumor**. Klinisch wäre das inakzeptabel. (Die exakten Zahlen können je nach sklearn-Version minimal abweichen.)

## ⭐ 6. Ein zweites Modell: Logistische Regression

Die **logistische Regression** kennen Sie vielleicht aus der Statistik. Sie ist in der Medizin weit verbreitet (z. B. für Risiko-Scores) und gibt **Wahrscheinlichkeiten** aus.

Ein Detail: Die Merkmale haben sehr unterschiedliche Größenordnungen (z. B. *area* ≈ 500, *smoothness* ≈ 0,1). Viele Modelle funktionieren besser, wenn alle Merkmale vorher **standardisiert** werden (Mittelwert 0, SD 1 – wie z-Werte). Eine **Pipeline** verbindet Standardisierung und Modell zu einem Objekt, sodass die Standardisierung nur aus den Trainingsdaten gelernt wird.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)
vorhersage_logreg = logreg.predict(X_test)

vergleich = pd.DataFrame({
    "Entscheidungsbaum": guetekriterien(y_test, vorhersage_baum),
    "Log. Regression":   guetekriterien(y_test, vorhersage_logreg),
})
vergleich.round(3)

Beachten Sie: Wir mussten fast nichts am Code ändern – nur die Modellklasse ausgetauscht. Das ist die große Stärke von sklearn.

## ⭐ 7. Wahrscheinlichkeiten und der Schwellenwert

`predict` entscheidet intern: *Wahrscheinlichkeit für bösartig ≥ 0,5 → bösartig*. Mit `predict_proba` sehen wir die Wahrscheinlichkeiten selbst:

In [ ]:
wahrscheinlichkeit = logreg.predict_proba(X_test)[:, 1]   # Spalte 1 = P(bösartig)

pd.DataFrame({
    "P(bösartig)": wahrscheinlichkeit[:8].round(3),
    "wahre Diagnose": y_test.values[:8],
})

Der Schwellenwert 0,5 ist willkürlich! In einem **Screening** will man möglichst keinen Krebsfall übersehen und nimmt dafür mehr falsch-positive Befunde in Kauf. Senkt man den Schwellenwert, steigt die Sensitivität – auf Kosten der Spezifität:

In [ ]:
ergebnisse = {}
for schwelle in [0.1, 0.3, 0.5, 0.7, 0.9]:
    vorhersage = (wahrscheinlichkeit >= schwelle).astype(int)
    ergebnisse[f"Schwelle {schwelle}"] = guetekriterien(y_test, vorhersage)

pd.DataFrame(ergebnisse).round(3)

> Die Wahl des Schwellenwerts ist **keine statistische, sondern eine klinische Entscheidung**: Was ist schlimmer – ein übersehener Tumor oder eine unnötige Biopsie?

## 8. ROC-Kurve und AUC

Die **ROC-Kurve** zeigt Sensitivität gegen (1 − Spezifität) für *alle* möglichen Schwellenwerte. Die **Fläche unter der Kurve (AUC)** fasst die Trennschärfe in einer Zahl zusammen: 0,5 = Münzwurf, 1,0 = perfekt.

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(logreg, X_test, y_test, name="Log. Regression", ax=ax)
RocCurveDisplay.from_estimator(baum, X_test, y_test, name="Entscheidungsbaum", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Zufall (AUC = 0,5)")
ax.set_xlabel("1 − Spezifität (Falsch-positiv-Rate)")
ax.set_ylabel("Sensitivität (Richtig-positiv-Rate)")
ax.legend()
plt.show()

## 9. Overfitting: Wenn das Modell auswendig lernt

Ein Entscheidungsbaum mit unbegrenzter Tiefe kann die Trainingsdaten **perfekt** auswendig lernen – inklusive aller Zufälligkeiten. Auf neuen Daten wird er dadurch aber nicht besser, oft sogar schlechter. Das nennt man **Overfitting**.

In [ ]:
tiefen = range(1, 16)
train_genauigkeit, test_genauigkeit = [], []

for tiefe in tiefen:
    m = DecisionTreeClassifier(max_depth=tiefe, random_state=42).fit(X_train, y_train)
    train_genauigkeit.append(m.score(X_train, y_train))
    test_genauigkeit.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tiefen, train_genauigkeit, marker="o", label="Trainingsdaten")
ax.plot(tiefen, test_genauigkeit, marker="o", label="Testdaten")
ax.set_xlabel("Maximale Tiefe des Baums")
ax.set_ylabel("Genauigkeit")
ax.set_title("Overfitting: Training ≠ Realität")
ax.legend()
plt.show()

Die Trainingsgenauigkeit erreicht ab einer gewissen Tiefe 100 %, die Testgenauigkeit bleibt deutlich darunter und verbessert sich nicht weiter. **Hohe Trainingsgenauigkeit allein beweist gar nichts.**

## 10. Kreuzvalidierung: robustere Bewertung

Ein einziger Test-Split kann Glück oder Pech sein. Bei der **k-fachen Kreuzvalidierung** werden die Trainingsdaten in k Teile geteilt; das Modell wird k-mal trainiert, jedes Mal wird ein anderer Teil zum Testen benutzt.

In [ ]:
from sklearn.model_selection import cross_val_score

auc_werte = cross_val_score(logreg, X_train, y_train, cv=5, scoring="roc_auc")
print("AUC je Durchgang:", auc_werte.round(3))
print(f"Mittlere AUC: {auc_werte.mean():.3f} (SD {auc_werte.std():.3f})")

## ⭐ 11. Grenzen und kritische Fragen

Ein Modell mit über 95 % Genauigkeit auf einem Lehrdatensatz ist **noch lange kein Diagnosewerkzeug**. Fragen, die Sie bei jeder KI-Studie stellen sollten:

- **Externe Validierung:** Wurde das Modell an Daten aus *einer anderen Klinik / einem anderen Gerät* getestet? Modelle versagen oft, wenn sich die Population ändert.
- **Prävalenz:** PPV und NPV hängen von der Prävalenz ab. Im Testdatensatz sind ca. 37 % bösartig – im Screening wäre die Prävalenz viel niedriger, der PPV damit deutlich schlechter.
- **Data Leakage:** Sind Informationen aus den Testdaten ins Training „durchgesickert“ (z. B. dieselbe Patientin in beiden Teilen, Standardisierung vor dem Split)?
- **Bias:** Ist die Trainingspopulation repräsentativ (Alter, Ethnie, Geschlecht, Krankheitsstadien)?
- **Klinischer Nutzen:** Verbessert das Modell tatsächlich Entscheidungen und Outcomes – oder nur eine Kennzahl?

Für die Bewertung von Vorhersagemodell-Studien gibt es Leitlinien wie **TRIPOD+AI** – ein Blick lohnt sich, wenn Sie selbst zu dem Thema promovieren.

---
## 📝 Übungsaufgaben

### Aufgabe 1: Ein weiteres Modell
Probieren Sie einen **Random Forest** (viele Entscheidungsbäume, die „abstimmen“) aus. Importieren Sie ihn mit `from sklearn.ensemble import RandomForestClassifier`, trainieren Sie ihn mit `n_estimators=200, random_state=42` und vergleichen Sie Sensitivität und Spezifität mit den anderen Modellen.

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
from sklearn.ensemble import RandomForestClassifier

wald = RandomForestClassifier(n_estimators=200, random_state=42)
wald.fit(X_train, y_train)
vorhersage_wald = wald.predict(X_test)

vergleich["Random Forest"] = guetekriterien(y_test, vorhersage_wald)
vergleich.round(3)
```
Random Forests brauchen keine Standardisierung, da Bäume nur Schwellenwerte vergleichen.
</details>

### Aufgabe 2: Screening-Schwelle finden
Finden Sie für die logistische Regression den **höchsten** Schwellenwert, bei dem im Testdatensatz **kein einziger** bösartiger Tumor übersehen wird (Sensitivität = 100 %). Wie hoch ist dann die Spezifität?

*Tipp:* Testen Sie Schwellenwerte mit `np.arange(0.01, 1.0, 0.01)` in einer Schleife.

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
beste_schwelle = None
for schwelle in np.arange(0.01, 1.0, 0.01):
    vorhersage = (wahrscheinlichkeit >= schwelle).astype(int)
    if guetekriterien(y_test, vorhersage)["Sensitivität"] == 1.0:
        beste_schwelle = schwelle

print(f"Höchste Schwelle mit Sensitivität 100 %: {beste_schwelle:.2f}")
print(guetekriterien(y_test, (wahrscheinlichkeit >= beste_schwelle).astype(int)).round(3))
```
**Achtung:** Diese Schwelle wurde *auf den Testdaten* gesucht. Streng genommen sind die Testdaten damit „verbraucht“ – in einer echten Studie würde man den Schwellenwert auf den Trainingsdaten (per Kreuzvalidierung) festlegen und erst dann einmalig testen.
</details>

### Aufgabe 3 (Transfer): Diabetes im eigenen Datensatz vorhersagen
Laden Sie `patienten.csv` aus Notebook 02 und trainieren Sie eine logistische Regression, die `diabetes` aus `alter`, `groesse_cm`, `gewicht_kg`, `sys_rr` und `raucher` vorhersagt. Entfernen Sie vorher Zeilen mit fehlenden Werten. Wie hoch ist die AUC auf den Testdaten?

*Tipp:* `from sklearn.metrics import roc_auc_score` und `roc_auc_score(y_test, wahrscheinlichkeiten)`.

In [ ]:
# Ihre Lösung hier

<details>
<summary><b>Lösung anzeigen</b></summary>

```python
from sklearn.metrics import roc_auc_score

pat = pd.read_csv("patienten.csv")
merkmale = ["alter", "groesse_cm", "gewicht_kg", "sys_rr", "raucher"]
pat = pat.dropna(subset=merkmale + ["diabetes"])

Xp, yp = pat[merkmale], pat["diabetes"].astype(int)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    Xp, yp, test_size=0.3, stratify=yp, random_state=0)

modell = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modell.fit(Xp_train, yp_train)
p = modell.predict_proba(Xp_test)[:, 1]
print(f"AUC: {roc_auc_score(yp_test, p):.3f}")
```
Mit nur 200 Patient\*innen und ca. 20 Diabetesfällen ist das Ergebnis sehr unsicher – probieren Sie verschiedene `random_state`-Werte aus und beobachten Sie, wie stark die AUC schwankt. Genau deshalb braucht man große Datensätze und Kreuzvalidierung!
</details>

---
## 📌 Zusammenfassung

| Schritt | Code |
|---|---|
| Aufteilen | `train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)` |
| Modell erzeugen | `DecisionTreeClassifier(max_depth=3)`, `LogisticRegression()`, `RandomForestClassifier()` |
| Vorverarbeitung + Modell | `make_pipeline(StandardScaler(), LogisticRegression())` |
| Trainieren | `modell.fit(X_train, y_train)` |
| Vorhersagen | `modell.predict(X_test)`, `modell.predict_proba(X_test)[:, 1]` |
| Bewerten | `accuracy_score`, `confusion_matrix`, `RocCurveDisplay`, `cross_val_score` |

### Die wichtigsten Botschaften
1. **Niemals** auf den Trainingsdaten bewerten.
2. Genauigkeit allein reicht nicht – **Sensitivität und Spezifität** sind klinisch entscheidend.
3. Der **Schwellenwert** ist eine klinische Entscheidung.
4. Komplexere Modelle sind nicht automatisch besser (**Overfitting**).
5. Ein gutes Ergebnis auf einem Datensatz ist kein Beleg für klinischen Nutzen (**externe Validierung**).

🎉 **Herzlichen Glückwunsch!** Sie haben jetzt die vier wichtigsten Werkzeuge der Datenanalyse mit Python kennengelernt.